# FaceLab Analysis

This notebook compares **Human** vs **AI** faces in a simple, consistent way:
- Part 1: **accuracy** and **reaction time**
- Part 2: **ratings**

Every plot follows the same pattern: make a summary table first, then plot that table.
Error bars show an approximate **95% confidence interval**: mean +/- 1.96 * standard error.


In [ ]:
from pathlib import Path
from IPython import get_ipython
from IPython.display import display

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import PercentFormatter

ip = get_ipython()
if ip is not None:
    ip.run_line_magic("matplotlib", "inline")

sns.set_theme(style="whitegrid")

FACE_ORDER = ["Human", "AI"]
EMOTION_ORDER = ["happy", "sad", "angry", "fear"]
PALETTE = {"Human": "#2ca25f", "AI": "#f28e2b"}
# Error bars are approximate 95% confidence intervals: mean +/- 1.96 * standard error.
CI_MULTIPLIER = 1.96


## Parameters

Change `START_DATE` if you only want to analyze data from a certain date onward.


In [ ]:
START_DATE = None  # Example: "2026-03-01"
ONLY_CURRENT_EMOTIONS = True


## Helpers

These small helpers keep the notebook short while making the calculations explicit.


In [ ]:
def pick_file(preferred_name, pattern):
    path = Path(preferred_name)
    if path.exists():
        return path
    matches = sorted(Path(".").glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Could not find {preferred_name}.")
    return matches[-1]


def summarize_mean_ci(df, group_cols, value_col, scale=1):
    summary = (
        df.groupby(group_cols)[value_col]
        .agg(N="size", Mean="mean", SD="std")
        .reset_index()
    )
    summary["SD"] = summary["SD"].fillna(0)
    summary["SE"] = summary["SD"] / summary["N"].pow(0.5)
    summary["CI"] = CI_MULTIPLIER * summary["SE"]
    summary["Mean"] = summary["Mean"] * scale
    summary["CI"] = summary["CI"] * scale
    return summary.drop(columns=["SD", "SE"])


def plot_simple_bars(ax, table, x_col, y_col, err_col, order, title, ylabel, percent=False):
    plot_df = table.set_index(x_col).reindex(order).reset_index()
    ax.bar(
        plot_df[x_col],
        plot_df[y_col],
        yerr=plot_df[err_col],
        color=[PALETTE[x] for x in plot_df[x_col]],
        capsize=4,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.margins(y=0.18)
    if percent:
        ax.yaxis.set_major_formatter(PercentFormatter())
    for x, y in zip(plot_df[x_col], plot_df[y_col]):
        label = f"{y:.1f}%" if percent else f"{y:.0f}"
        ax.text(x, y, label, ha="center", va="bottom")
    sns.despine(ax=ax)


def plot_grouped_bars(ax, table, x_col, hue_col, y_col, err_col, x_order, hue_order, title, ylabel, percent=False, rotate=0):
    width = 0.35
    x_positions = list(range(len(x_order)))

    for i, hue in enumerate(hue_order):
        subset = table[table[hue_col] == hue].set_index(x_col).reindex(x_order)
        offset = (i - (len(hue_order) - 1) / 2) * width
        bar_positions = [x + offset for x in x_positions]
        ax.bar(
            bar_positions,
            subset[y_col],
            width=width,
            yerr=subset[err_col],
            capsize=4,
            color=PALETTE[hue],
            label=hue,
        )

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_order, rotation=rotate, ha="right" if rotate else "center")
    if percent:
        ax.yaxis.set_major_formatter(PercentFormatter())
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.16), ncol=2, frameon=False, title=None)
    sns.despine(ax=ax)


## Part 1 Data

Part 1 contains the emotion-choice outcomes, so this is where accuracy and reaction time come from.


In [ ]:
part1_path = pick_file("emotion_responses_part1.csv", "emotion_responses_part1*.csv")
part1_df = pd.read_csv(part1_path)

needed = {"stimulus_type", "target_emotion", "emotion_rt_ms", "accuracy", "emotion_timestamp"}
missing = needed - set(part1_df.columns)
if missing:
    raise ValueError(f"Missing Part 1 columns: {sorted(missing)}")

part1_df["emotion_timestamp"] = pd.to_datetime(part1_df["emotion_timestamp"], errors="coerce")
part1_df["emotion_rt_ms"] = pd.to_numeric(part1_df["emotion_rt_ms"], errors="coerce")
part1_df["face_source"] = part1_df["stimulus_type"].map({"real_kdef": "Human", "ai_kdef_like": "AI"})
part1_df["emotion"] = (
    part1_df["target_emotion"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"afraid": "fear", "fearful": "fear"})
)
part1_df["accuracy_num"] = (
    part1_df["accuracy"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"correct": 1, "incorrect": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0, "true": 1, "false": 0})
)

analysis_df = part1_df[part1_df["face_source"].isin(FACE_ORDER)].copy()
if ONLY_CURRENT_EMOTIONS:
    analysis_df = analysis_df[analysis_df["emotion"].isin(EMOTION_ORDER)].copy()
if START_DATE:
    analysis_df = analysis_df[analysis_df["emotion_timestamp"] >= pd.Timestamp(START_DATE)].copy()

analysis_df = analysis_df.dropna(subset=["emotion_timestamp", "emotion_rt_ms", "accuracy_num", "emotion"])

if analysis_df.empty:
    raise ValueError("No Part 1 rows left after filtering. Try an earlier START_DATE.")

display(pd.DataFrame({
    "Rows used": [len(analysis_df)],
    "Start": [analysis_df["emotion_timestamp"].min()],
    "End": [analysis_df["emotion_timestamp"].max()],
    "File": [part1_path.name],
}))


## Part 1 Overall Summary


In [ ]:
accuracy_overall = summarize_mean_ci(analysis_df, ["face_source"], "accuracy_num", scale=100).rename(
    columns={"Mean": "Accuracy (%)", "CI": "Accuracy CI"}
)
rt_overall = summarize_mean_ci(analysis_df, ["face_source"], "emotion_rt_ms").rename(
    columns={"Mean": "Mean RT (ms)", "CI": "RT CI"}
)

summary = (
    accuracy_overall.merge(rt_overall[["face_source", "Mean RT (ms)", "RT CI"]], on="face_source")
    .rename(columns={"face_source": "Face", "N": "Trials"})
    .set_index("Face")
    .reindex(FACE_ORDER)
    .reset_index()
)

summary[["Accuracy (%)", "Accuracy CI", "Mean RT (ms)", "RT CI"]] = summary[["Accuracy (%)", "Accuracy CI", "Mean RT (ms)", "RT CI"]].round(2)
display(summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

plot_simple_bars(
    axes[0],
    summary,
    x_col="Face",
    y_col="Accuracy (%)",
    err_col="Accuracy CI",
    order=FACE_ORDER,
    title="Accuracy: Human vs AI",
    ylabel="Percent correct",
    percent=True,
)

plot_simple_bars(
    axes[1],
    summary,
    x_col="Face",
    y_col="Mean RT (ms)",
    err_col="RT CI",
    order=FACE_ORDER,
    title="Reaction Time: Human vs AI",
    ylabel="Mean RT (ms)",
)

plt.tight_layout()
plt.show()


## Part 1 By Emotion


In [ ]:
accuracy_by_emotion = summarize_mean_ci(analysis_df, ["emotion", "face_source"], "accuracy_num", scale=100).rename(
    columns={"Mean": "Accuracy (%)", "CI": "Accuracy CI"}
)
rt_by_emotion = summarize_mean_ci(analysis_df, ["emotion", "face_source"], "emotion_rt_ms").rename(
    columns={"Mean": "Mean RT (ms)", "CI": "RT CI"}
)

by_emotion = (
    accuracy_by_emotion.merge(
        rt_by_emotion[["emotion", "face_source", "Mean RT (ms)", "RT CI"]],
        on=["emotion", "face_source"],
    )
    .rename(columns={"emotion": "Emotion", "face_source": "Face", "N": "Trials"})
)

by_emotion["Emotion"] = pd.Categorical(by_emotion["Emotion"], categories=EMOTION_ORDER, ordered=True)
by_emotion["Face"] = pd.Categorical(by_emotion["Face"], categories=FACE_ORDER, ordered=True)
by_emotion = by_emotion.sort_values(["Emotion", "Face"]).reset_index(drop=True)
by_emotion[["Accuracy (%)", "Accuracy CI", "Mean RT (ms)", "RT CI"]] = by_emotion[["Accuracy (%)", "Accuracy CI", "Mean RT (ms)", "RT CI"]].round(2)
display(by_emotion)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_grouped_bars(
    axes[0],
    by_emotion,
    x_col="Emotion",
    hue_col="Face",
    y_col="Accuracy (%)",
    err_col="Accuracy CI",
    x_order=EMOTION_ORDER,
    hue_order=FACE_ORDER,
    title="Accuracy By Emotion",
    ylabel="Percent correct",
    percent=True,
)

plot_grouped_bars(
    axes[1],
    by_emotion,
    x_col="Emotion",
    hue_col="Face",
    y_col="Mean RT (ms)",
    err_col="RT CI",
    x_order=EMOTION_ORDER,
    hue_order=FACE_ORDER,
    title="Reaction Time By Emotion",
    ylabel="Mean RT (ms)",
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


## Part 2 Ratings

Part 2 is handled in exactly the same way: summarize first, then plot the summary.


In [ ]:
part2_path = pick_file("emotion_responses_part2.csv", "emotion_responses_part2*.csv")
part2_df = pd.read_csv(part2_path)

rating_cols = [
    "match_age_rating",
    "match_masc_rating",
    "match_attr_rating",
    "match_quality_rating",
    "match_artifact_rating",
]
rating_labels = {
    "match_age_rating": "Age match",
    "match_masc_rating": "Masculinity",
    "match_attr_rating": "Attractiveness",
    "match_quality_rating": "Image quality",
    "match_artifact_rating": "Artifacts",
}

needed = {"stimulus_type", "matching_timestamp", *rating_cols}
missing = needed - set(part2_df.columns)
if missing:
    raise ValueError(f"Missing Part 2 columns: {sorted(missing)}")

part2_df["matching_timestamp"] = pd.to_datetime(part2_df["matching_timestamp"], errors="coerce")
part2_df["face_source"] = part2_df["stimulus_type"].map({"real_kdef": "Human", "ai_kdef_like": "AI"})
for col in rating_cols:
    part2_df[col] = pd.to_numeric(part2_df[col], errors="coerce")

part2_df = part2_df[part2_df["face_source"].isin(FACE_ORDER)].copy()
if START_DATE:
    part2_df = part2_df[part2_df["matching_timestamp"] >= pd.Timestamp(START_DATE)].copy()
part2_df = part2_df.dropna(subset=["matching_timestamp"])

if part2_df.empty:
    raise ValueError("No Part 2 rows left after filtering. Try an earlier START_DATE.")

part2_long = part2_df.melt(
    id_vars="face_source",
    value_vars=rating_cols,
    var_name="Measure",
    value_name="Rating",
).dropna(subset=["Rating"])
part2_long["Measure"] = part2_long["Measure"].map(rating_labels)

part2_summary = summarize_mean_ci(part2_long, ["Measure", "face_source"], "Rating").rename(
    columns={"Measure": "Measure", "face_source": "Face", "Mean": "Average rating", "CI": "Rating CI", "N": "Trials"}
)
part2_summary["Measure"] = pd.Categorical(part2_summary["Measure"], categories=list(rating_labels.values()), ordered=True)
part2_summary["Face"] = pd.Categorical(part2_summary["Face"], categories=FACE_ORDER, ordered=True)
part2_summary = part2_summary.sort_values(["Measure", "Face"]).reset_index(drop=True)
part2_summary[["Average rating", "Rating CI"]] = part2_summary[["Average rating", "Rating CI"]].round(2)
display(pd.DataFrame({
    "Rows used": [len(part2_df)],
    "Start": [part2_df["matching_timestamp"].min()],
    "End": [part2_df["matching_timestamp"].max()],
    "File": [part2_path.name],
}))
display(part2_summary)


In [ ]:
plt.figure(figsize=(11, 4.8))
ax = plt.gca()

plot_grouped_bars(
    ax,
    part2_summary,
    x_col="Measure",
    hue_col="Face",
    y_col="Average rating",
    err_col="Rating CI",
    x_order=list(rating_labels.values()),
    hue_order=FACE_ORDER,
    title="Part 2 Ratings: Human vs AI",
    ylabel="Average rating",
    rotate=15,
)
ax.set_ylim(0, 5.6)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()
